<!--
SPDX-FileCopyrightText: Copyright (c) 2026, NVIDIA CORPORATION & AFFILIATES.
All rights reserved.
SPDX-License-Identifier: Apache-2.0
-->

# ViDoRe v3 HR: Ingestion and Retrieval Evaluation

An exmaple showing how to ingest the Vidore v3 HR corpus into LanceDB, retrieve relevant pages, and report **Recall and nDCG at 1, 3, 5, and 10** with NeMo Retriever Library.

Use a **Python 3.12** kernel. For first-time setup, enable dependency installation and run only the prep cell. Restart the kernel, disable installation, then edit the configuration and choose **Run All Cells**. Local mode requires NVIDIA GPUs and sufficient memory for extraction and embedding models. Hosted mode uses remote HTTP models and needs no GPU.

## 1. Prepare the environment

Start with a Python 3.12 environment following the
[installation guide](https://github.com/NVIDIA/NeMo-Retriever/tree/main/nemo_retriever#setup-your-environment).
Local inference needs **Linux, CUDA 13.x, and NVIDIA GPUs**. Hosted inference needs no CUDA.

Choose `LOCAL_MODELS=True` for local models or `False` for hosted HTTP models.
The prep cell uses `uv` to install into the active kernel.
For a new environment, set `INSTALL_DEPENDENCIES=True` and run **only this cell** once.
It installs from PyPI and, for local mode, replaces Torch/Torchvision with the
CUDA 13 builds specified in the guide. Restart the kernel afterward and set `INSTALL_DEPENDENCIES=False` before choosing **Run All Cells**.
Hosted NVIDIA endpoints require `NVIDIA_API_KEY` in the kernel environment.

In [ ]:
import sys

LOCAL_MODELS = True
INSTALL_DEPENDENCIES = False

if INSTALL_DEPENDENCIES:
    package = "nemo-retriever[local]" if LOCAL_MODELS else "nemo-retriever"
    !uv pip install --python "{sys.executable}" "{package}" "datasets>=4.8.2" "huggingface-hub>=1.5,<2"
    if LOCAL_MODELS:
        !uv pip uninstall --python "{sys.executable}" torch torchvision
        !uv pip install --python "{sys.executable}" torch==2.11.0 torchvision==0.26.0 --index-url https://download.pytorch.org/whl/cu130

## 2. Configure the run

Set `MAX_QUERIES=20` for a quick retrieval check or `None` for the full test split.
The entire corpus is indexed in either case. Local mode requires enough GPU memory
for extraction and embedding; hosted mode sends pages and questions to the endpoints below.

Data and index directories are created in the current working directory.
Indexes use one directory per mode. **Rebuild after changing models, endpoints, or corpus**
by setting `OVERWRITE_INDEX=True`. The hosted URLs can point to compatible NIM deployments.

In [ ]:
from pathlib import Path

MAX_QUERIES = None
OVERWRITE_INDEX = False

EMBED_MODEL = "nvidia/Nemotron-3-Embed-1B-BF16" if LOCAL_MODELS else "nvidia/nemotron-3-embed-1b"
PAGE_ELEMENTS_URL = "https://ai.api.nvidia.com/v1/cv/nvidia/nemotron-page-elements-v3"
OCR_URL = "https://ai.api.nvidia.com/v1/cv/nvidia/nemotron-ocr-v2"
TABLE_STRUCTURE_URL = "https://ai.api.nvidia.com/v1/cv/nvidia/nemotron-table-structure-v1"
EMBED_URL = "https://integrate.api.nvidia.com/v1/embeddings"

MODE = "local" if LOCAL_MODELS else "hosted"
CORPUS_DIR = Path("vidore_v3_hr_data") / "pdfs"
INDEX_DIR = Path(f"lancedb-vidore-v3-hr-{MODE}")
TABLE_NAME = "vidore_v3_hr"
assert MAX_QUERIES is None or MAX_QUERIES > 0

## 3. Prepare the dataset

Load [ViDoRe v3 HR](https://huggingface.co/datasets/vidore/vidore_v3_hr)'s test split.
The BEIR helper loads questions and relevance labels, keyed by corpus page ID.
Hugging Face caches the download; set `HF_TOKEN` or log in if access requires authentication.

Export each corpus image to `<corpus_id>.pdf` so retrieved filenames match the labels.
We ingest the page images rather than the reference markdown. Existing PDFs are reused.

In [ ]:
from datasets import load_dataset
from tqdm.auto import tqdm
from nemo_retriever.tools.recall.beir import load_beir_dataset

corpus = load_dataset("vidore/vidore_v3_hr", data_dir="corpus", split="test")
# Keep corpus page IDs as the relevance-label keys.
dataset = load_beir_dataset("vidore_hf", dataset_name="vidore_v3_hr", doc_id_field="source_id")
query_ids = dataset.query_ids[:MAX_QUERIES]
queries = dataset.queries[:MAX_QUERIES]
qrels = {qid: dataset.qrels[qid] for qid in query_ids}
corpus_ids = {str(value) for value in corpus["corpus_id"]}
assert corpus_ids and queries, "The corpus and questions must not be empty."

CORPUS_DIR.mkdir(parents=True, exist_ok=True)
for page in tqdm(corpus, desc="Preparing pages"):
    path = CORPUS_DIR / f"{page['corpus_id']}.pdf"
    if not path.exists():
        page["image"].convert("RGB").save(path, format="PDF", resolution=150.0)

print(f"{len(corpus):,} corpus pages · {len(queries):,} questions")
print(f"Example question: {queries[0]}")

## 4. Ingest the pages

Use **PDFium hybrid** to extract text from native PDF text or OCR of page images,
plus **tables, charts, and infographics**.
Then embed the combined text of each page and store one vector per page in LanceDB.
The first ingestion can take a while; an existing table is reused unless
`OVERWRITE_INDEX=True`. Rebuild an older index to apply these extraction and embedding settings.

Local mode uses local models and vLLM embedding backend. Hosted mode
routes Page Elements, OCR, table structure, and embedding to your configured endpoints.
If ingestion is interrupted, rebuild the index.
Routine logs, worker stdout/stderr, and progress bars are suppressed without saving a log file.
Failures still raise exceptions. Restart the kernel once if an earlier Ray session
was started with worker log forwarding enabled.

Pages with no extractable content may be skipped; a lower count can also indicate incomplete ingestion.


In [ ]:
import logging
import os
from contextlib import redirect_stdout, redirect_stderr
import lancedb
from ray.data import DataContext
from nemo_retriever import create_ingestor

# Keep notebook output brief; failures still raise exceptions.
logging.disable(logging.WARNING)
DataContext.get_current().enable_progress_bars = False

index = lancedb.connect(str(INDEX_DIR))
table_exists = TABLE_NAME in index.list_tables(limit=None).tables
if OVERWRITE_INDEX or not table_exists:
    print("Ingesting pages…")
    extract_options = {}
    embed_options = {}
    if not LOCAL_MODELS:
        extract_options = {
            "page_elements_invoke_url": PAGE_ELEMENTS_URL,
            "ocr_invoke_url": OCR_URL,
            "table_structure_invoke_url": TABLE_STRUCTURE_URL,
        }
        embed_options = {"embed_invoke_url": EMBED_URL}

    with open(os.devnull, "w") as quiet, redirect_stdout(quiet), redirect_stderr(quiet):
        (
            create_ingestor(
                run_mode="batch", allow_no_gpu=not LOCAL_MODELS, ray_log_to_driver=False
            )
            .files([str(path) for path in CORPUS_DIR.glob("*.pdf")])
            .extract(
                method="pdfium_hybrid",
                extract_text=True,
                extract_tables=True,
                extract_charts=True,
                extract_infographics=True,
                use_table_structure=True,
                **extract_options,
            )
            .embed(
                embed_model_name=EMBED_MODEL,
                embed_granularity="page",
                embed_modality="text",
                **embed_options,
            )
            .vdb_upload(
                vdb_kwargs={
                    "uri": str(INDEX_DIR),
                    "table_name": TABLE_NAME,
                    "overwrite": OVERWRITE_INDEX,
                }
            )
            .ingest()
        )
    index = lancedb.connect(str(INDEX_DIR))
else:
    print("Reusing the existing index.")

indexed_pages = index.open_table(TABLE_NAME).count_rows()
assert indexed_pages > 0, "Empty index: set OVERWRITE_INDEX=True."
print(f"Indexed {indexed_pages}/{len(corpus)} pages.")

## 5. Retrieve relevant pages

Embed each question with the same model used for ingestion and retrieve the top ten pages.
Each indexed vector represents one page with a unique corpus ID, so no extra
candidate search or deduplication is needed. The BEIR helper converts the results
into the format used for evaluation. Short or empty rankings still count in the scores.

In [ ]:
import os
from nemo_retriever.graph.retriever import Retriever
from nemo_retriever.tools.recall.beir import build_beir_run_from_hits

embed_kwargs = {"model_name": EMBED_MODEL, "embed_model_name": EMBED_MODEL}
if not LOCAL_MODELS:
    embed_kwargs.update(embedding_endpoint=EMBED_URL, api_key=os.environ.get("NVIDIA_API_KEY", ""))

retriever = Retriever(
    run_mode="local" if LOCAL_MODELS else "service",
    vdb_kwargs={"uri": str(INDEX_DIR), "table_name": TABLE_NAME},
    embed_kwargs=embed_kwargs,
)
hits = retriever.queries(queries, top_k=10)
assert all(
    hit.get("pdf_basename") in corpus_ids for query_hits in hits for hit in query_hits
), "A retrieved page ID does not match the corpus. Check the index or rebuild it."
run = build_beir_run_from_hits(query_ids, hits, doc_id_field="pdf_basename")
short_rankings = sum(len(ranking) < 10 for ranking in run.values())
print(f"Retrieved pages for {len(run):,} questions.")

## 6. Evaluate retrieval

**Recall@k** measures the fraction of relevant pages found in the first k results.
**nDCG@k** rewards putting the most relevant pages first, using the labels' relevance
grades and a logarithmic rank discount. Higher is better for both metrics.

The table averages scores over every selected question, including empty rankings.
`summary_df` holds the scores; `run` holds page rankings and `hits` holds retrieved content.

In [ ]:
import pandas as pd
from IPython.display import display
from nemo_retriever.tools.recall.beir import compute_beir_metrics

cutoffs = (1, 3, 5, 10)
metrics = compute_beir_metrics(qrels, run, ks=cutoffs)
summary_df = pd.DataFrame(
    {
        "Recall": [metrics[f"recall@{k}"] for k in cutoffs],
        "nDCG": [metrics[f"ndcg@{k}"] for k in cutoffs],
    },
    index=pd.Index(cutoffs, name="k"),
)
display(summary_df.style.format("{:.3f}"))